# 🏭 ArtisanAI Studio

## Gere imagensincríveis com IA!


In [ ]:
#@title 📝 1. Configuração
projeto = "logo"  #@param {type:"string"}
descricao = "Logo moderno azul para logistics"  #@param {type:"string"}
quantas = 3  #@param {type:"integer", min:1, max:10}
modo = "basic"  #@param ["basic", "pro"]
modelo_local = ""  #@param {type:"string", label: "Pasta local (opicional)"}

In [ ]:
#@title ⚙️ 2. Baixar Projeto + Instalar
%cd /content
import urllib.request, zipfile, shutil, os
urllib.request.urlretrieve(
    "https://github.com/zelu-undo/Design-auto/archive/refs/heads/main.zip", "main.zip")
with zipfile.ZipFile("main.zip", 'r') as z:
    z.extractall(".")
shutil.move("Design-auto-main", "Design-auto")
os.remove("main.zip")
%cd /content/Design-auto
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install -q diffusers transformers accelerate
print("✅ OK!")

In [ ]:
#@title 📥 Baixar Modelo (só 1ª vez)
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download
pasta_modelo = "modelo_cache"
if modelo_local:
    pasta_modelo = modelo_local
elif not os.path.exists(pasta_modelo):
    print("📦 Baixando modelo (~2GB)...")
    snapshot_download(
        repo_id="stabilityai/sdxl-turbo",
        local_dir=pasta_modelo,
        local_use_symlinks=False
    )
    print(f"✅ Salvo em: {pasta_modelo}")
else:
    print(f"✅ Modelo já existe: {pasta_modelo}")
print(f"modelo_local = '{pasta_modelo}'")

In [ ]:
#@title 🎨 3. Gerar
import sys, os
sys.path.insert(0, '/content/Design-auto')
os.chdir('/content/Design-auto')
from src import Orquestrador
orc = Orquestrador(projeto, modo=modo, modelo_local=pasta_modelo)
res = orc.executar_pipeline_completo(descricao, num_imagens=quantas)
qtd = res.get('estatisticas', {}).get('total_imagens_aprovadas', 0)
print(f"\n✅ {qtd} imagens salvas!")

In [ ]:
#@title 🖼️ 4. Ver
from IPython.display import display, Image, Markdown
pasta = f"projetos/{projeto}/aprovadas"
if os.path.exists(pasta):
    arqs = sorted([f for f in os.listdir(pasta) if f.endswith(('.png', '.jpg'))])
    display(Markdown(f"### 🎉 {len(arqs)} Imagens"))
    for f in arqs:
        display(Image(f"{pasta}/{f}", width=400))

In [ ]:
#@title 📥 5. Baixar
import shutil
from google.colab import files
shutil.make_archive(projeto, 'zip', 'projetos', projeto)
files.download(f"{projeto}.zip")